# PLY viewer

Pick a grounding failure the same way [`failure_explorer.ipynb`](failure_explorer.ipynb)
does — the full failure table, grouped by cause, worst IoU first — and draw its scene as
a **solid mesh** with the boxes over it.

| box | colour |
|---|---|
| ground truth | green |
| prediction | red |
| other objects of the same class | grey |


In [ ]:
# ---- config --------------------------------------------------------------------------
PREDICTIONS   = "outputs/2024-12-18_20-40-38_3DVG-FIXED/predictions.p"
SPLIT         = "val"
SCANS_ROOT    = None         # None uses data/scannet/scans

# ---- which failures to list ----------------------------------------------------------
CAUSES        = []           # e.g. ["small_object", "distractor_confusion"]; [] = all
SCENES        = []           # e.g. ["scene0011_00"]; [] = all
IOU_MIN       = None
IOU_MAX       = None
PICK_LIMIT    = 0            # rows listed per cause; 0 lists every one

# Leave empty to be prompted. Set it to re-draw a known case without a prompt:
#   SELECTION = "7"
SELECTION     = ""

# ---- look ----------------------------------------------------------------------------
MESH_OPACITY  = 1         # <1 makes walls translucent so boxes inside stay visible
BOX_WIDTH     = 6            # cage line thickness
GAMMA         = 0.75         # <1 lifts midtones; ScanNet scans are captured dark
MAX_FACES     = 0            # 0 = every triangle; set e.g. 200000 to decimate a slow scene
SHOW_DISTRACTORS = True      # grey cages on every other instance of the target's class
RENDERER      = "browser"       # "auto" draws inline in VSCode/Jupyter
CAMERA_EYE    = (1.4, -1.4, 0.9)
HEIGHT        = 750

# ---- failure definition (matches `failure-cases` defaults) ---------------------------
THRESHOLD        = 0.25      # a failure is IoU below this
CRITERION        = "fuzzy"
PARSE_FOLDER     = "final_parsing_tokenized"
SMALL_PERCENTILE = 10.0
LONG_PERCENTILE  = 75.0
MANY_SPATIAL     = 3
DISTRACTOR_IOU   = 0.05
DRIFT_LOW        = 0.10

In [ ]:
import os, sys, pickle
from argparse import Namespace
from collections import Counter

import numpy as np

if not os.path.isfile("experiments/analysis.py"):       # be runnable from a subdirectory
    os.chdir(os.path.dirname(os.path.abspath("")) or ".")
sys.path.insert(0, os.getcwd())

# The failure table, the picker, the mesh loader and the box colours are all the
# analysis script's -- same code failure_explorer and the CLI run.
from experiments.analysis import (
    load_scanrefer, load_predictions, join, load_parse_cache, parse_for,
    scene_class_counts, attribute, spatial_hits, box_volume, match_kinds,
    filter_failures, pick_cases, md_table, same_class_boxes, normalise_corners,
    load_scan_ply, axis_alignment, _BOX_EDGES,
    GT_COLOR, PRED_COLOR, DISTRACTOR_COLOR, SCANNET_SCANS,
    CAUSES as ALL_CAUSES, CAUSE_TEXT,
)

# Plotly's default renderer here is "browser", which opens an external window instead of
# drawing in the notebook -- so figure.show() appears to do nothing inline.
import plotly.graph_objects as go
import plotly.io as pio
if RENDERER == "auto":
    pio.renderers.default = "vscode+notebook_connected+browser"
elif RENDERER:
    pio.renderers.default = RENDERER
print(f"cwd {os.getcwd()}")
print(f"plotly renderer: {pio.renderers.default}")


def read_scan_mesh(scene_id, scans_root=None):
    """(xyz, rgb, faces) for a scan, aligned into the box frame. Triangles kept.

    load_scan_ply returns points only; the figures never needed the faces. Here they are
    the whole point, so the file is read again for its face element and the same
    axisAlignment matrix is applied to the vertices.
    """
    from plyfile import PlyData
    root = scans_root or SCANNET_SCANS
    path = os.path.join(root, scene_id, f"{scene_id}_vh_clean_2.ply")
    if not os.path.isfile(path):
        return None, None, None, path
    data = PlyData.read(path)
    v = data["vertex"].data
    xyz = np.stack([v["x"], v["y"], v["z"]], axis=1).astype(np.float32)
    rgb = np.stack([v["red"], v["green"], v["blue"]], axis=1).astype(np.uint8)
    matrix = axis_alignment(scene_id, root)
    if matrix is not None:
        xyz = (np.concatenate([xyz, np.ones((len(xyz), 1), np.float32)], axis=1)
               @ matrix.T)[:, :3].astype(np.float32)
    else:
        print(f"  ! {scene_id}: no axisAlignment -- boxes will not line up")
    raw = data["face"].data
    faces = np.stack(raw[raw.dtype.names[0]]).astype(np.int32)
    return xyz, rgb, faces, path


def box_lines(corners):
    """The 12 edges as one x/y/z triple with None between segments.

    One Scatter3d trace holds a whole disconnected cage this way; a trace per edge would
    be twelve legend entries and twelve times the payload.
    """
    corners = normalise_corners(corners)
    xs, ys, zs = [], [], []
    for a, b in _BOX_EDGES:
        xs += [corners[a][0], corners[b][0], None]
        ys += [corners[a][1], corners[b][1], None]
        zs += [corners[a][2], corners[b][2], None]
    return xs, ys, zs

In [ ]:
# ---- build the failure table ---------------------------------------------------------
# The same computation `failure-cases` runs, kept in memory so the picker can work on it.
ARGS = Namespace(criterion=CRITERION, distractor_iou=DISTRACTOR_IOU,
                 drift_low=DRIFT_LOW, many_spatial=MANY_SPATIAL)

records = load_scanrefer(SPLIT)
rows, missing = join(load_predictions(PREDICTIONS), records)
print(f"{len(rows)} annotations joined" + (f"  ({missing} without a prediction)" if missing else ""))

try:
    cache = load_parse_cache(PARSE_FOLDER, SPLIT)
except FileNotFoundError as error:
    print(f"WARNING: {error}\n         parse-based causes will be unavailable")
    cache = {}

counts = scene_class_counts(records)
small_threshold = float(np.percentile(
    [box_volume(r["_prediction"]["gt_bbox"]) for r in rows], SMALL_PERCENTILE))
long_threshold = float(np.percentile(
    [len(r.get("token") or []) for r in rows], LONG_PERCENTILE))

failures = []
for row in rows:
    if row["iou"] >= THRESHOLD:
        continue
    parsed = parse_for(cache, row["scene_id"], row["object_id"], row["ann_id"]) or {}
    predicted_target = " ".join(parsed.get("target") or [])
    target_ok = (match_kinds(predicted_target, row["object_name"])[CRITERION]
                 if parsed else True)
    signals = {
        "parse_target_ok": bool(target_ok),
        "same_class_instances": counts[row["scene_id"]].get(row["object_name"], 1),
        "gt_volume": round(box_volume(row["_prediction"]["gt_bbox"]), 4),
        "small_threshold": round(small_threshold, 4),
        "num_tokens": int(len(row.get("token") or [])),
        "long_threshold": int(long_threshold),
        "num_spatial": len(spatial_hits(row.get("token") or [])),
    }
    failures.append({
        "scene_id": row["scene_id"], "object_id": row["object_id"],
        "ann_id": row["ann_id"], "object_name": row["object_name"],
        "description": row.get("description", ""),
        "iou": row["iou"], "cause": attribute(row, signals, ARGS), "signals": signals,
        "parsed_target": predicted_target or "(no parse)",
        "parsed_adjectives": " ".join(parsed.get("adjectives") or []) or "(no parse)",
        "parsed_neighbors": " ".join(parsed.get("neighbors") or []) or "(no parse)",
    })

tally = Counter(c["cause"] for c in failures)
print(f"\n{len(failures)} of {len(rows)} annotations "
      f"({100.0 * len(failures) / max(len(rows), 1):.2f}%) fall below IoU {THRESHOLD}\n")
print(md_table(["cause", "count", "% of failures"],
               [[c, tally[c], f"{100.0 * tally[c] / max(len(failures), 1):.1f}"]
                for c in ALL_CAUSES if tally[c]]))

In [ ]:
# ---- list the candidates and pick ----------------------------------------------------
# Grouped by cause, each group worst-IoU first. Indices run globally, so "3" or "17"
# addresses any row. Set SELECTION in the config cell to skip the prompt.
pool = filter_failures(failures, CAUSES, SCENES, IOU_MIN, IOU_MAX)
if len(pool) != len(failures):
    print(f"[filter] {len(pool)} of {len(failures)} failures match\n")

# top=1: this notebook draws one scene at a time.
chosen = pick_cases(pool, PICK_LIMIT, SELECTION if SELECTION else None, top=1) if pool else []
CASE = chosen[0] if chosen else None

if CASE:
    print(f"\n{CASE['cause']}  IoU={CASE['iou']:.3f}  {CASE['scene_id']} "
          f"obj{CASE['object_id']} ({CASE['object_name']}) ann{CASE['ann_id']}")
    print(f"  \"{CASE['description']}\"")
    print(f"  why: {CAUSE_TEXT.get(CASE['cause'], '')}")
    print(f"  parsed target={CASE['parsed_target']!r}  "
          f"adjectives={CASE['parsed_adjectives']!r}  "
          f"neighbours={CASE['parsed_neighbors']!r}")
else:
    print("nothing selected -- run this cell again to pick")

In [ ]:
# ---- draw the scene ------------------------------------------------------------------
if not CASE:
    raise SystemExit("no case selected -- run the picker cell first")

XYZ, RGB, FACES, scan_path = read_scan_mesh(CASE["scene_id"], SCANS_ROOT)
if XYZ is None:
    raise SystemExit(f"no scan found at {scan_path}")
print(f"{len(XYZ):,} vertices  {len(FACES):,} triangles  {os.path.relpath(scan_path)}")

if MAX_FACES and len(FACES) > MAX_FACES:
    FACES = FACES[np.linspace(0, len(FACES) - 1, MAX_FACES).astype(int)]
    print(f"  decimated to {len(FACES):,} triangles (MAX_FACES)")

with open(PREDICTIONS, "rb") as handle:
    entry = pickle.load(handle)[CASE["scene_id"]][str(CASE["object_id"])][str(CASE["ann_id"])]

BOXES = []
if SHOW_DISTRACTORS:
    for other in same_class_boxes(CASE["scene_id"], CASE["object_id"]):
        BOXES.append(("same-class instance", other, DISTRACTOR_COLOR))
BOXES.append(("ground truth", np.asarray(entry["gt_bbox"], float), GT_COLOR))
BOXES.append(("prediction", np.asarray(entry["pred_bbox"], float), PRED_COLOR))

# Mesh3d takes per-vertex colour as a uint8 array -- unlike Scatter3d, which needs
# "#rrggbb" strings and silently falls back to a colorscale if handed an array.
shaded = (np.clip(RGB / 255.0, 0, 1) ** GAMMA * 255).astype(np.uint8)
traces = [go.Mesh3d(x=XYZ[:, 0], y=XYZ[:, 1], z=XYZ[:, 2],
                    i=FACES[:, 0], j=FACES[:, 1], k=FACES[:, 2],
                    vertexcolor=shaded, opacity=MESH_OPACITY, flatshading=False,
                    lighting=dict(ambient=0.6, diffuse=0.6, specular=0.05, roughness=0.9),
                    hoverinfo="skip", showlegend=False, name="scan")]

seen = set()
for label, corners, color in BOXES:
    xs, ys, zs = box_lines(corners)
    traces.append(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                               line=dict(color=color, width=BOX_WIDTH),
                               name=label, legendgroup=label,
                               showlegend=label not in seen, hoverinfo="skip"))
    seen.add(label)

figure = go.Figure(traces)
figure.update_layout(
    margin=dict(l=0, r=0, t=34, b=0), height=HEIGHT,
    title=(f"{CASE['cause']}  --  {CASE['scene_id']}  object {CASE['object_id']}  "
           f"annotation {CASE['ann_id']} ({CASE['object_name']})  --  IoU {CASE['iou']:.3f}"),
    legend=dict(orientation="h", yanchor="bottom", y=0.01, x=0.01),
    # aspectmode="data" keeps the room's real proportions; "cube" would pad every axis
    # out to the largest extent and shrink the room to a speck in the middle.
    scene=dict(aspectmode="data",
               camera=dict(eye=dict(x=CAMERA_EYE[0], y=CAMERA_EYE[1], z=CAMERA_EYE[2])),
               xaxis=dict(title="x (m)"), yaxis=dict(title="y (m)"),
               zaxis=dict(title="z (m)")))
figure.show()

In [ ]:
# ---- optional: save what is on screen -------------------------------------------------
# .html orbits in any browser with no notebook; .png needs kaleido.
SAVE_TO = ""          # e.g. "outputs/analysis/case_view.html"; "" to skip

if SAVE_TO:
    os.makedirs(os.path.dirname(os.path.abspath(SAVE_TO)) or ".", exist_ok=True)
    if SAVE_TO.endswith(".html"):
        figure.write_html(SAVE_TO, include_plotlyjs="cdn")
    else:
        figure.write_image(SAVE_TO, width=1400, height=HEIGHT)
    print(f"wrote {SAVE_TO}  ({os.path.getsize(SAVE_TO) / 1e6:.1f} MB)")